# AI Agents in Action: Building an Amazon Review Analyst

Welcome to the world of AI Agents! An AI agent is more than just a model; it's a system designed to achieve a goal by creating a plan and taking actions. It can reason, use tools, and adapt its approach to solve complex problems.

In this notebook, we will build a simplified but powerful **Shopping Assistant Agent**. Its goal will be to analyze a large dataset of Amazon food reviews to provide a tailored product recommendation based on a user's specific needs.

We will simulate the agent's workflow step-by-step:

1.  **Part 1: The Goal**: We'll define a clear, specific goal for our agent.
2.  **Part 2: The Plan (Reasoning)**: We'll use an LLM to look at the goal and create a logical, step-by-step plan to achieve it using the available data.
3.  **Part 3: The Execution (Action)**: We will execute the plan, using both traditional data analysis with Pandas and LLM calls to analyze unstructured review text.
4.  **Part 4: The Conclusion**: Our agent will synthesize its findings into a final, actionable recommendation for the user.

## Learning Objectives
- Understand the core workflow of an AI agent: **Goal -> Plan -> Action -> Conclusion**.
- Use a large language model (LLM) to generate a logical **plan of action**.
- Combine structured data analysis (**Pandas**) with unstructured text analysis (**LLM**).
- Implement an LLM-powered summarization and analysis of customer reviews.
- Synthesize multiple pieces of information into a final, data-driven recommendation.

---

## Part 1: Setup & The Goal

First, we'll set up our environment by installing libraries, configuring our API key, and loading the dataset. Most importantly, we'll define a clear goal for our shopping assistant agent.

In [1]:
# Install necessary libraries
!pip install -q -U google-generativeai pandas

import pandas as pd
import google.generativeai as genai
from google.colab import userdata
import warnings
warnings.filterwarnings('ignore')

# Configure the Gemini API Key (store it as 'GEMINI_API_KEY' in Colab Secrets)
try:
    api_key = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=api_key)
    print("API key configured successfully! ✅")
except userdata.SecretNotFoundError as e:
    print("Secret not found. Please add your GEMINI_API_KEY to Colab Secrets.")

# Initialize the Generative Model
model = genai.GenerativeModel('gemini-1.5-flash-latest')

# Load the dataset (using a smaller sample for speed)
# Make sure 'AmazonFoodReviews.csv' is uploaded to your Colab session
try:
    df = pd.read_csv('AmazonFoodReviews.csv', nrows=20000) # Use 20k reviews for this demo
    print(f"Successfully loaded {len(df)} reviews. ✅")
except FileNotFoundError:
    print("ERROR: 'AmazonFoodReviews.csv' not found. Please upload the file.")

# ----------------------------------
#  THE AGENT'S GOAL
# ----------------------------------
USER_GOAL = """
I am looking for the best coffee beans for making espresso. 
I need something with a rich, dark roast, low acidity, and reviewers should mention that it produces a good 'crema'.
"""

print("\nAgent's Goal has been defined:")
print(USER_GOAL)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.16.2 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 5.29.5 which is incompatible.
streamlit 1.32.0 requires protobuf<5,>=3.20, but you have protobuf 5.29.5 which is incompatible.


ModuleNotFoundError: No module named 'google.colab'

---

## Part 2: The Plan (Reasoning Step)

A key feature of an agent is its ability to **reason** and create a plan. We will ask the LLM to act as our agent and devise a strategy to accomplish the `USER_GOAL` using the `df` DataFrame.

In [ ]:
planning_prompt = f"""
You are an expert data analyst and research assistant AI agent. 
Your goal is to find the best product based on a user's request, using a pandas DataFrame named 'df' which contains Amazon reviews.
The DataFrame has columns including 'ProductId', 'Summary', and 'Text'.

Here is the user's goal: 
--- 
{USER_GOAL}
--- 

Create a concise, step-by-step plan to analyze the DataFrame and find the best product. Your plan should involve filtering the data, identifying relevant products, summarizing reviews, and making a final recommendation.
"""

print("Asking the agent to create a plan...")
response = model.generate_content(planning_prompt)
agent_plan = response.text

print("\n--- AGENT'S PLAN OF ACTION ---")
print(agent_plan)

This is powerful! The agent has thought through the problem and given us a clear, logical sequence of actions. Now, we'll execute this plan.

---

## Part 3: The Execution (Action Step)

Here, we'll follow the agent's plan. This involves a mix of coding (for filtering) and calling the LLM as a tool (for summarizing and analyzing).

In [ ]:
print("--- Step 1: Filter for relevant reviews (coffee) ---")
df_coffee = df[df['Text'].str.contains('coffee', case=False, na=False)].copy()
print(f"Found {len(df_coffee)} reviews containing the word 'coffee'.")

print("\n--- Step 2: Identify top candidate products ---")
# Find products that are frequently reviewed
top_products = df_coffee['ProductId'].value_counts().head(10)
print("Top 10 most reviewed coffee products:")
print(top_products)

top_product_ids = top_products.index.tolist()
df_top_coffee = df_coffee[df_coffee['ProductId'].isin(top_product_ids)]

In [ ]:
print("\n--- Step 3: Summarize and analyze reviews for top products ---")
product_summaries = {}

analysis_prompt_template = """
Analyze the following Amazon reviews for a coffee product. Based *only* on the text provided, determine if it meets these criteria:
1. Good for espresso
2. Dark roast with rich flavor
3. Low acidity
4. Produces good 'crema'

Provide a summary of pros and cons related to these criteria and conclude with a 'Suitability Score' from 0 to 10.

REVIEWS:
{reviews}
"""

for product_id in top_product_ids:
    print(f"\nAnalyzing Product: {product_id}...")
    # Get up to 5 reviews for the product
    reviews_text = "\n---\n".join(df_top_coffee[df_top_coffee['ProductId'] == product_id]['Text'].head(5).tolist())
    
    # Ask the LLM to analyze these reviews
    prompt = analysis_prompt_template.format(reviews=reviews_text)
    response = model.generate_content(prompt)
    product_summaries[product_id] = response.text
    print("Analysis complete.")
    
print("\n--- Example Summary for one product ---")
print(f"Product ID: {top_product_ids[0]}\n")
print(product_summaries[top_product_ids[0]])

---

## Part 4: The Conclusion (Final Recommendation)

Our agent has gathered and analyzed the data. The final step is to synthesize this information into a clear, user-friendly recommendation.

In [ ]:
final_summary_text = "\n\n".join([f"Product ID: {pid}\nAnalysis:\n{summary}" for pid, summary in product_summaries.items()])

final_prompt = f"""
You are a helpful shopping assistant. Based on the following analyses of several coffee products, make a final recommendation for a user with the following goal:

USER GOAL: {USER_GOAL}

ANALYSES:
--- 
{final_summary_text}
--- 

Present your final recommendation clearly. Announce a 'Top Pick' and explain exactly why it meets the user's criteria better than the others. You can also mention a runner-up if applicable.
"""

print("--- AGENT'S FINAL RECOMMENDATION ---")
response = model.generate_content(final_prompt)
print(response.text)

## Conclusion

In this notebook, you successfully simulated the workflow of an AI agent. You saw how a complex goal can be broken down into a **plan**, how that plan can be **executed** using a combination of code and AI tools, and how the results can be **synthesized** into a final, valuable output.

This Goal -> Plan -> Action -> Conclusion loop is the fundamental logic that powers sophisticated agentic AI systems. By making each step visible, you've gained a deep, practical understanding of how these systems can reason and solve problems to achieve specific business objectives.